# Hugging Face Agent Model Lab

Laboratorio externo; el producto funciona sin este notebook. No contiene tokens
ni salidas guardadas. Usa los contratos reales, pruebas semánticas y el adapter
OpenAI-compatible existente. No ejecuta código generado por modelos.

Ejecutar desde un checkout del proyecto, con un kernel del entorno local.
Para cloud GPU, provisionar un entorno separado con torch, transformers y
huggingface_hub; nunca se incluyen como dependencias obligatorias del producto.
Consultar docs/MODEL-LAB-REPORT.md antes de habilitar inferencia.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
from decimal import Decimal
from tools.model_lab import discover, shortlist, benchmark, save_results, recommend, GateRequired
from tools.model_lab_cases import CASES


## A. Inference Providers — descubrimiento público

Solo GET de metadatos, sin token y sin generar texto. Los campos ausentes quedan
desconocidos. La preselección por metadatos no es una recomendación validada.


In [ ]:
catalog = discover()
candidates = shortlist(catalog, limit=3)
[{"routing_id": c.routing_id, "context": c.context_length,
  "input_usd_per_million": c.input_usd_per_million,
  "output_usd_per_million": c.output_usd_per_million,
  "structured_advertised": c.supports_structured_output,
  "tools_advertised": c.supports_tools} for c in candidates]


## Gate de credenciales y consumo

HF_TOKEN se configura fuera del notebook: entorno o .env local ignorado por Git.
No imprimirlo, pegarlo en celdas ni guardar outputs. La existencia del token no
autoriza gasto. El saldo incluido de la cuenta no se deduce del catálogo.

`AUTHORIZED_USD` permanece None hasta la autorización expresa del usuario.
El guard reserva contexto completo + salida máxima por cada intento previsto,
sin reembolso por fallo. Un límite de cuenta del proveedor es necesario para
un techo monetario contractual externo. No hay reintentos ocultos ni fallbacks.


In [ ]:
import os
from dotenv import dotenv_values
AUTHORIZED_USD = None  # se establece solo tras autorización económica explícita
RUN_REMOTE = False
if RUN_REMOTE:
    credential = os.environ.get("HF_TOKEN") or dotenv_values(ROOT / ".env").get("HF_TOKEN") or ""
    try:
        measurements = benchmark(candidates, token=credential, max_usd=AUTHORIZED_USD)
        save_results(ROOT / "var/model_lab_results.json", candidates, measurements)
        routing = recommend(candidates, measurements)
    finally:
        credential = None
else:
    measurements, routing = [], {}


## Interpretación

8 casos: Analyst, Supervisor, Coder y Reviewer, 2 repeticiones mínimas por caso.
Researcher continúa determinista. Se separan JSON válido, esquema válido y tarea
correcta. Usage ausente y coste no fiable son null. No se declara compatibilidad
de tool calls por una marca de catálogo: el contrato runtime no expone tools.
Una recomendación requiere todos los casos del rol correctos en dos repeticiones.
La batería pequeña produce evidencia preliminar, no un benchmark general.


In [ ]:
{"attempts": len(measurements), "routing_recommendation": routing}


## B. GPU cloud opcional

No se ejecuta en el runtime ni automáticamente. Requiere entorno GPU preparado y
acceso/licencia del modelo. Usar ID y SHA completo actuales del Hub, no un alias
mutable. La función evita trust_remote_code, exige safetensors y bloquea pesos
por encima de 2 GB o tamaño desconocido. La cuota/coste de GPU cloud se autoriza
separadamente. Los resultados no equivalen a la validación del proveedor remoto.


In [ ]:
RUN_CLOUD = False
CLOUD_MODEL_ID = None
CLOUD_REVISION = None
if RUN_CLOUD:
    if not CLOUD_MODEL_ID or not CLOUD_REVISION:
        raise GateRequired("modelo y revisión fijados, hardware y gasto cloud autorizados requeridos")
    from tools.model_lab_cloud import benchmark_cloud
    cloud_measurements = benchmark_cloud(CLOUD_MODEL_ID, CLOUD_REVISION)


## Exportación

Solo métricas y metadatos permitidos. Nunca exportar tokens, headers, variables
de entorno o contenido bruto de respuestas. Limpiar todas las salidas antes de
versionar el notebook. Fuentes oficiales y estado real en MODEL-LAB-REPORT.md.
